# 🚀 Microsoft phi-4-mini & SQLite FTS5 (சூனிய மேகம் SLM) உடன் ஹைபிரிட் RAG  

> **எழுத்தாளர்:** Çağrı Giray Keşan ([@Cagrik34](https://github.com/Cagrik34))  
> **முகப்பு:** சிறிய மொழி மாதிரிகள் (SLMs), SQLite FTS5 BM25, அடர்த்தியான எம்பெட்டிங்குகள், பரஸ்பர தரவரிசை கரம் (RRF)  

---

## 📌 1. ஊக்கமருத்துவம்: உள்ளூர் SLMகளில் முக்கிய வார்த்தை நினைவுப் பிரச்சனை  
முழுமையாக அடர்த்தியான வெக்டர் எம்பெட்டிங்குகளுக்கு மட்டுமே அ основыபடும் சாதாரண RAG கட்டமைப்புகள் சரியான எண்கணக்குக் குறியீடுகளை (எ.கா., `2,340,000 TL`, ஒப்பந்தக் குறியீடுகள், கணக்கு எண்கள்) மீட்டெடுக்க முறியடைகின்றன.  
மாறாக, குறைக்கப்பட்ட சொற்றொடர் தேடல் (BM25) கருத்தாற்றலின் இணைப்புக்களையும் மறுபடி அமைந்த கேள்விகளையும் தவறவிட்டுக் கொள்கிறது.  

இந்த குக்க்புக் ஒரு **உயர்தர மற்றும் நினைவகத்தில் இயங்கும் ஹைபிரிட் மறுபடியாய்த் தேடல் இயந்திரம்** உருவாக்குவதற்கான முறையை காட்டுகிறது, இதில் சேர்க்கப்பட்டுள்ளது:  
1. **அடர்த்தியான வெக்டர்கள்** (கோசைன் ஒத்திருப்பு)  
2. **குறைவான சொற்களின் தேடல்** (SQLite FTS5 BM25)  
3. **பரஸ்பர தரவரிசை கரம் (RRF, $k=60$)**  
4. Microsoft `phi-4-mini` உடன் **அடிப்படையான மேற்கோள் உள்வாங்கல் (`[1]`, `[2]`)**  


In [ ]:
import os
import sqlite3
import numpy as np
from typing import List, Tuple, Dict, Any

RRF_K = 60
TOP_K = 2
print("✅ Core dependencies loaded successfully.")

## 🏗️ 2. இரட்டை SQLite திட்டம் (அடர்த்தியான வெக்டார்கள் + மெய்நிகர் FTS5 BM25 அட்டவணை)


In [ ]:
class LocalHybridRAGStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    chunk_index INTEGER NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                )
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    chunk_index UNINDEXED,
                    tokenize='unicode61'
                )
            """)

    def insert_chunk(self, source_file: str, chunk_index: int, content: str, embedding: List[float]) -> None:
        vec = np.array(embedding, dtype=np.float32)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm

        with self.conn:
            self.conn.execute(
                "INSERT INTO document_chunks (source_file, chunk_index, content, embedding) VALUES (?, ?, ?, ?)",
                (source_file, chunk_index, content, vec.tobytes())
            )
            self.conn.execute(
                "INSERT INTO document_chunks_fts (content, source_file, chunk_index) VALUES (?, ?, ?)",
                (content, source_file, str(chunk_index))
            )

    def search_dense(self, query_embedding: List[float], top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        q_vec = np.array(query_embedding, dtype=np.float32)
        q_norm = np.linalg.norm(q_vec)
        if q_norm > 0:
            q_vec = q_vec / q_norm

        cursor = self.conn.execute("SELECT id, source_file, content, embedding FROM document_chunks")
        results = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            similarity = float(np.dot(q_vec, doc_vec))
            results.append((doc_id, src, content, similarity))
        results.sort(key=lambda x: x[3], reverse=True)
        return results[:top_k]

    def search_sparse_bm25(self, query_text: str, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        clean_tokens = [t for t in query_text.replace("'", "").replace('"', '').split() if len(t) > 1]
        if not clean_tokens:
            return []
        fts_query = " OR ".join(f'"{t}"' for t in clean_tokens)
        cursor = self.conn.execute(
            "SELECT rowid, source_file, content, rank FROM document_chunks_fts WHERE document_chunks_fts MATCH ? ORDER BY rank LIMIT ?",
            (fts_query, top_k)
        )
        results = []
        for doc_id, src, content, bm25_rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(bm25_rank)))
            results.append((doc_id, src, content, bm25_score))
        return results

    def hybrid_search(self, query_text: str, query_embedding: List[float], top_k: int = TOP_K) -> List[Dict[str, Any]]:
        dense_hits = self.search_dense(query_embedding, top_k=10)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=10)
        fused_scores = {}
        chunk_map = {}

        for rank, (doc_id, src, content, sim) in enumerate(dense_hits, start=1):
            key = f"{src}::{content[:50]}"
            chunk_map[key] = (src, content, "vector")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (RRF_K + rank))

        for rank, (doc_id, src, content, bm25) in enumerate(sparse_hits, start=1):
            key = f"{src}::{content[:50]}"
            if key not in chunk_map:
                chunk_map[key] = (src, content, "bm25")
            else:
                chunk_map[key] = (src, content, "hybrid")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (RRF_K + rank))

        sorted_keys = sorted(fused_scores.keys(), key=lambda k: fused_scores[k], reverse=True)[:top_k]
        output = []
        for citation_idx, key in enumerate(sorted_keys, start=1):
            src, content, match_type = chunk_map[key]
            output.append({
                "citation_index": citation_idx,
                "source_file": src,
                "content": content,
                "rrf_score": fused_scores[key],
                "match_type": match_type
            })
        return output

print("✅ LocalHybridRAGStore class compiled successfully.")

## 📊 3. மாதிரி சேர்க்கை மற்றும் செயலாக்க ஒப்புமுறை


In [ ]:
store = LocalHybridRAGStore()

sample_docs = [
    ("q3_financial_report.pdf", 0, "CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers.", [0.8, 0.1, 0.2] + [0.0] * 1021),
    ("architecture_specs.md", 0, "Zenith AI leverages Microsoft phi-4-mini (3.8B parameters) for local zero-cloud inference.", [0.2, 0.9, 0.1] + [0.0] * 1021),
    ("hr_policy_2026.docx", 0, "Remote work expense allowance is capped at 15,000 TL per employee quarterly.", [0.1, 0.1, 0.8] + [0.0] * 1021)
]

for src, idx, content, emb in sample_docs:
    store.insert_chunk(src, idx, content, emb)

query = "What is the total allocated budget for the CodePulse project?"
query_vec = [0.75, 0.15, 0.25] + [0.0] * 1021

results = store.hybrid_search(query, query_vec, top_k=2)
for res in results:
    print(f"[{res['citation_index']}] {res['source_file']} ({res['match_type'].upper()}) -> Score: {res['rrf_score']:.4f}")
    print(f"    Content: {res['content']}\n")

## 📝 4. மைக்ரோசாஃப்ட் phi-4-mini க்கான நிலைத்த(prompt) முன்மொழிவு வடிவமைப்பு


In [ ]:
def construct_grounded_prompt(query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for chunk in retrieved_chunks:
        context_blocks.append(f"[{chunk['citation_index']}] (Source: {chunk['source_file']})\n{chunk['content']}")
    context_str = "\n\n".join(context_blocks)

    return f"""You are Zenith AI, an enterprise-grade local assistant.
Answer the user query strictly based on the provided context below.
Every factual claim must cite its source index like [1] or [2].
If the context does not contain the answer, respond: 'This information is not present in the indexed documents.'

--- CONTEXT ---
{context_str}
--- END CONTEXT ---

User Query: {query}
Answer:"""

prompt = construct_grounded_prompt(query, results)
print(prompt)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
